# Magenta RealTime 2 on Kaggle (2x T4)


Two-cell setup using the curl|bash installer (venv-isolated, so Kaggle's
system packages are not touched), then sharded generation. Run on a Kaggle
notebook with the GPU accelerator enabled (2x T4).


## 1. Install (clones fork, creates a venv, downloads + quantizes)


In [ ]:
# Installs into an isolated venv at /kaggle/working/mrt_venv so Kaggle's
# RAPIDS/pandas/jupyter-server are NOT touched. Idempotent; safe to re-run.
# To use your own fork: set MRT_REPO=... before the curl.
!curl -fsSL https://raw.githubusercontent.com/ctunix/magenta-realtime/main/scripts/install_kaggle.sh | bash


## 2. Generate 8 s of audio (sharded across 2 GPUs if present) + play


In [ ]:
# Uses the venv mrt with JAX memory env vars set BEFORE the Python process starts.
# --shard shards the 2.4B model across all local CUDA GPUs (falls back to 1 GPU).
!XLA_PYTHON_CLIENT_PREALLOCATE=false XLA_PYTHON_CLIENT_MEM_FRACTION=0.85 MAGENTA_HOME=/kaggle/working /kaggle/working/mrt_venv/bin/mrt jax generate --model mrt2_base --checkpoint mrt2_base_bf16.safetensors --shard --duration 8 --prompt "disco funk"


In [ ]:
# Play the result.
import IPython.display as ipd
ipd.display(ipd.Audio('/kaggle/working/magenta-rt-v2/outputs/output_audio_jax_mrt2_base.wav', rate=48000))


## Notes


- The installer sets `MAGENTA_HOME=/kaggle/working` (the **base** dir);
`paths.py` appends `magenta-rt-v2` itself, so assets land in
`/kaggle/working/magenta-rt-v2/`. (Setting `.../magenta-rt-v2` would double-nest.)
- On Colab, replace `/kaggle/working` with `/content`.
- To keep the fp32 checkpoint (e.g. for fp32 + 2-GPU sharding), re-run with
`KEEP_FP32=1 curl ... | bash` and use `--checkpoint mrt2_base.safetensors`.
